In [ ]:
!pip install fuzzywuzzy python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.2 MB/s eta 0:00:00


All smell descriptors found in the Good Scents company website

In [2]:
import pandas as pd

smell_descriptors = pd.read_csv("smell_descriptors.csv").iloc[:, 0].tolist()
print(smell_descriptors[:5])

['absinthe', 'acacia', 'acai', 'acerola', 'acetic']


# Pre-processing the datasets

## Arctander

In [ ]:
import pandas as pd

df = pd.read_csv("arctander.csv")
df = df[["SMILES", "ChastretteDetails"]]

In [ ]:
df = df.dropna().reset_index(drop=True)
df

,SMILES,ChastretteDetails
0,CC(C)C1CCC2C(=C1)CCC3C2(CCCC3(C)CO)C,woody piney
1,C(C)=O,ethereal winey
2,CC(OCCOC)OCC1=CC=CC=C1,green fruity
3,CC(C)CCOC(C)OCCC(C)C,oily green
4,CCOC(C)OCC,fruity green
...,...,...
2767,C12(C=CC=CC1(C)O2)C,almond
2768,CC1=CC(=C(C=C1)C=O)C,floral almond
2769,C1(=C(C(=CC=C1)C)CCO)C,floral balsamic rose
2770,O=C(C)CCC1=CC(=C(O)C=C1)OC,spicy floral animal balsamic vanillin


Correcting the little typos in the descriptors of the arctander dataset to match the correct ones that are present in the smell_descriptors list

In [ ]:
all_chastrette_words = df['ChastretteDetails'].str.split(' ').explode().unique().tolist()
print(all_chastrette_words[:10])

['woody', 'piney', 'ethereal', 'winey', 'green', 'fruity', 'oily', 'floral', 'herbaceous', 'rose']


In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 68
Common descriptors (first 10): ['tobacco', 'narcissus', 'orris', 'musty', 'rooty', 'vanilla', 'buttery', 'metallic', 'waxy', 'earthy']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in ChastretteDetails but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to ChastretteDetails (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in ChastretteDetails but not in smell_descriptors: 37
Descriptors unique to ChastretteDetails (first 10): ['', 'caramelic', 'must', 'gas', 'ambre', 'aldehidic', 'herbaceous', 'mint', 'musky', 'lillac']


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 70

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 35 potential transformations (threshold >= 70%):
  'caramelic' -> 'caramellic'
  'must' -> 'mustard'
  'gas' -> 'gasoline'
  'ambre' -> 'ambrette'
  'aldehidic' -> 'aldehydic'
  'herbaceous' -> 'herbal'
  'mint' -> 'cornmint'
  'musky' -> 'musk'
  'lillac' -> 'lilac'
  'pepper' -> 'peppery'


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split(' ')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['ChastretteDetails'] = df['ChastretteDetails'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'ChastretteDetails' after transformations:")
print(df['ChastretteDetails'].head())

First 5 rows of 'ChastretteDetails' after transformations:
0        woody pine
1    ethereal winey
2      green fruity
3        oily green
4      fruity green
Name: ChastretteDetails, dtype: object


Re-evaluating the unique words in `ChastretteDetails` and comparing them again with `smell_descriptors`

In [ ]:
all_chastrette_words_after_transform = df['ChastretteDetails'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_transform = list(set(all_chastrette_words_after_transform) - set(smell_descriptors))

print(f"Number of descriptors in ChastretteDetails but not in smell_descriptors (after transformations): {len(unique_chastrette_only_after_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_transform[:10]}")

Number of descriptors in ChastretteDetails but not in smell_descriptors (after transformations): 3
Remaining unique descriptors (first 10): ['', 'root', 'vanlilin']


Manually adding transformations for the remaining terms

In [ ]:
manual_transformations = {
    '': '',
    'root': 'rooty',
    'vanlilin': 'vanillin'
}

transformations.update(manual_transformations)

Re-applying `apply_transformations` function to the `ChastretteDetails` column with the updated `transformations`

In [ ]:
df['ChastretteDetails'] = df['ChastretteDetails'].apply(lambda x: apply_transformations(x, transformations))

all_chastrette_words_after_second_transform = df['ChastretteDetails'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_second_transform = list(set(all_chastrette_words_after_second_transform) - set(smell_descriptors))

print(f"Number of descriptors in ChastretteDetails but not in smell_descriptors (after second round of transformations): {len(unique_chastrette_only_after_second_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_second_transform[:10]}")

Number of descriptors in ChastretteDetails but not in smell_descriptors (after second round of transformations): 2
Remaining unique descriptors (first 10): ['', 'vanillin']


Separating the odor descriptors in columns based in all the labels available in `smell_descriptors` list

In [ ]:
for descriptor in smell_descriptors:
    df[descriptor] = df['ChastretteDetails'].str.contains(descriptor, case=False, na=False).astype(int)

/tmp/ipykernel_3427/3166053466.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['ChastretteDetails'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_3427/3166053466.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['ChastretteDetails'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_3427/3166053466.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [ ]:
df.head()

,SMILES,ChastretteDetails,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(C)C1CCC2C(=C1)CCC3C2(CCCC3(C)CO)C,woody pine,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C(C)=O,ethereal winey,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC(OCCOC)OCC1=CC=CC=C1,green fruity,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)CCOC(C)OCCC(C)C,oily green,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CCOC(C)OCC,fruity green,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("arctander_processed.csv", index=False)

## Flavordb

In [ ]:
import pandas as pd

df_molecules = pd.read_csv("molecules_flavordb.csv")
df_molecules = df_molecules[["CID", "IsomericSMILES"]]
df_molecules.head()

,CID,IsomericSMILES
0,4,CC(CN)O
1,40,C1C(C(C(OC1O)CO)O)O
2,47,CCC(C)C(=O)C(=O)O
3,49,CC(C)C(=O)C(=O)O
4,51,C(CC(=O)O)C(=O)C(=O)O


In [ ]:
df_behavior = pd.read_csv("behavior_flavordb.csv")
df_behavior = df_behavior[["Stimulus", "Flavor Percepts"]]
df_behavior.head()

,Stimulus,Flavor Percepts
0,4,fishy
1,40,sweet
2,47,fruity
3,49,fruity
4,51,odorless


In [ ]:
df = pd.merge(df_behavior, df_molecules, left_on='Stimulus', right_on='CID', how='left')
df = df.rename(columns={'IsomericSMILES': 'SMILES'})
df = df.drop(columns=['CID', 'Stimulus'])
df = df[["SMILES", "Flavor Percepts"]]
df = df.dropna().reset_index(drop=True)
df

,SMILES,Flavor Percepts
0,CC(CN)O,fishy
1,C1C(C(C(OC1O)CO)O)O,sweet
2,CCC(C)C(=O)C(=O)O,fruity
3,CC(C)C(=O)C(=O)O,fruity
4,C(CC(=O)O)C(=O)C(=O)O,odorless
...,...,...
25318,C[C@@]12CCC[C@@](C1CC[C@]34[C@H]2CCC(C3)(C(=C)...,sweet
25319,C(C1[C@H]([C@@H](C([C@H](O1)OCC(CO)O)O)O)O)O,sweet
25320,C[C@@]12CCC[C@@](C1CC[C@]34[C@H]2CC[C@](C3)(C(...,sweet
25321,C1=NC(=C2C(=N1)N(C=N2)C3[C@H]([C@H]([C@@H](O3)...,sweet


Correcting the little typos in the descriptors to match the correct ones that are present in the smell_descriptors list

In [ ]:
all_chastrette_words = df['Flavor Percepts'].str.split(';').explode().unique().tolist()
len(all_chastrette_words)

565

In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 325
Common descriptors (first 10): ['ozone', 'vetiver', 'herbal', 'lemon', 'vegetable', 'cooked', 'coconut', 'greasy', 'palmarosa', 'hyacinth']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Flavor Percepts but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Flavor Percepts (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Flavor Percepts but not in smell_descriptors: 240
Descriptors unique to Flavor Percepts (first 10): ['coumarin', 'caramel', 'flowery', 'mallow', 'pepper', 'parmesan', 'clothes', 'beet', 'acetate', 'amine']


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 70

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 168 potential transformations (threshold >= 70%):
  'coumarin' -> 'coumarinic'
  'caramel' -> 'caramellic'
  'flowery' -> 'cauliflower'
  'mallow' -> 'marshmallow'
  'pepper' -> 'peppery'
  'parmesan' -> 'cheesy parmesan cheese'
  'beet' -> 'beer'
  'acetate' -> 'acetone'
  'amine' -> 'jasmin'
  'needle' -> 'fir needle'


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split(' ')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['Flavor Percepts'] = df['Flavor Percepts'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Flavor Percepts' after transformations:")
print(df['Flavor Percepts'].head())

First 5 rows of 'Flavor Percepts' after transformations:
0       fishy
1       sweet
2      fruity
3      fruity
4    odorless
Name: Flavor Percepts, dtype: object


Re-evaluating the unique words in `Flavor Percepts` and comparing them again with `smell_descriptors`

In [ ]:
all_chastrette_words_after_transform = df['Flavor Percepts'].str.split(';').explode().unique().tolist()

unique_chastrette_only_after_transform = list(set(all_chastrette_words_after_transform) - set(smell_descriptors))

print(f"Number of descriptors in Flavor Percepts but not in smell_descriptors (after transformations): {len(unique_chastrette_only_after_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_transform[:10]}")

Number of descriptors in Flavor Percepts but not in smell_descriptors (after transformations): 238
Remaining unique descriptors (first 10): ['coumarin', 'caramel', 'flowery', 'mallow', 'pepper', 'parmesan', 'clothes', 'beet', 'acetate', 'amine']


Re-applying `apply_transformations` function to the `Flavor Percepts` column with the updated `transformations`

In [ ]:
df['Flavor Percepts'] = df['Flavor Percepts'].apply(lambda x: apply_transformations(x, transformations))

all_chastrette_words_after_second_transform = df['Flavor Percepts'].str.split(';').explode().unique().tolist()

unique_chastrette_only_after_second_transform = list(set(all_chastrette_words_after_second_transform) - set(smell_descriptors))

print(f"Number of descriptors in Flavor Percepts but not in smell_descriptors (after second round of transformations): {len(unique_chastrette_only_after_second_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_second_transform[:10]}")

Number of descriptors in Flavor Percepts but not in smell_descriptors (after second round of transformations): 239
Remaining unique descriptors (first 10): ['coumarin', 'caramel', 'flowery', 'mallow', 'pepper', 'parmesan', 'clothes', 'beet', 'acetate', 'amine']


Separating the odor descriptors in columns based in all the labels available in smell_descriptors list

In [ ]:
for descriptor in smell_descriptors:
    df[descriptor] = df['Flavor Percepts'].str.contains(descriptor, case=False, na=False).astype(int)

/tmp/ipykernel_2091/396452604.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Flavor Percepts'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/396452604.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Flavor Percepts'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/396452604.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfo

In [ ]:
df.head()

,SMILES,Flavor Percepts,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C1C(C(C(OC1O)CO)O)O,sweet,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CCC(C)C(=O)C(=O)O,fruity,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)C(=O)C(=O)O,fruity,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C(CC(=O)O)C(=O)C(=O)O,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("flavordb_processed.csv", index=False)

## Flavornet

In [ ]:
import pandas as pd

df_molecules = pd.read_csv("molecules_flavornet.csv")
df_molecules = df_molecules[["CID", "IsomericSMILES"]]
df_molecules.head()

,CID,IsomericSMILES
0,107,C1=CC=C(C=C1)CCC(=O)O
1,176,CC(=O)O
2,177,CC=O
3,179,CC(C(=O)C)O
4,240,C1=CC=C(C=C1)C=O


In [ ]:
df_behavior = pd.read_csv("behavior_flavornet.csv")
df_behavior.head()

,Stimulus,Descriptors
0,107,balsamic
1,176,sour
2,177,pungent;ether
3,179,cream;butter
4,240,burnt sugar;almond


In [ ]:
df = pd.merge(df_behavior, df_molecules, left_on='Stimulus', right_on='CID', how='left')
df = df.rename(columns={'IsomericSMILES': 'SMILES'})
df = df.drop(columns=['CID', 'Stimulus'])
df = df[["SMILES", "Descriptors"]]
df = df.dropna().reset_index(drop=True)
df

,SMILES,Descriptors
0,C1=CC=C(C=C1)CCC(=O)O,balsamic
1,CC(=O)O,sour
2,CC=O,pungent;ether
3,CC(C(=O)C)O,cream;butter
4,C1=CC=C(C=C1)C=O,burnt sugar;almond
...,...,...
711,CC1=C(CC(CC1)C(=C)C)OO,turpentine
712,CCCCCC(C=C)OO,metal;mushroom
713,CC1=CCC2CC1(C2(C)C)O,dust;must
714,CC(=CCC/C(=C\CCC(=C)/C=C/O)/C)C,oil;flower


Correcting the little typos in the descriptors to match the correct ones that are present in the `smell_descriptors` list

In [ ]:
all_chastrette_words = df['Descriptors'].str.split(';').explode().unique().tolist()
len(all_chastrette_words)

195

In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 101
Common descriptors (first 10): ['fresh', 'tallow', 'fried', 'paper', 'maple', 'honey', 'lemon', 'coconut', 'plastic', 'citrus']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Descriptors but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Descriptors (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors: 94
Descriptors unique to Descriptors (first 10): ['coumarin', 'caramel', 'tar', 'pepper', 'fruit', 'beet', 'amine', 'fish', 'dust', 'alcohol']


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 90

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 62 potential transformations (threshold >= 90%):
  'tar' -> 'custard'
  'pepper' -> 'peppery'
  'fruit' -> 'fruity'
  'fish' -> 'shellfish'
  'dust' -> 'sawdust'
  'fat' -> 'chicken fat'
  'leaf' -> 'tomato leaf'
  'cheese' -> 'cheesy bleu cheese'
  'roasted nut' -> 'roasted'
  'ether' -> 'ethereal'


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split(';')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after transformations:
0            balsamic
1                sour
2    pungent ethereal
3      creamy buttery
4        burnt almond
Name: Descriptors, dtype: object


In [ ]:
df['Descriptors']

,Descriptors
0,balsamic
1,sour
2,pungent ethereal
3,creamy buttery
4,burnt almond
...,...
711,turpentine
712,metallic mushroom
713,sawdust mustard
714,oil cauliflower


Re-evaluating the unique words in `Descriptors` and comparing them again with `smell_descriptors`

In [ ]:
all_chastrette_words_after_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_transform = list(set(all_chastrette_words_after_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after transformations): {len(unique_chastrette_only_after_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after transformations): 49
Remaining unique descriptors (first 10): ['coumarin', 'menthol', 'caramel', 'milk', 'pesticide', 'soap', 'beet', 'vinyl', 'cotton', 'bug']


Re-applying `apply_transformations` function to the `Descriptors` column with the updated `transformations`

In [ ]:
df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

all_chastrette_words_after_second_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_second_transform = list(set(all_chastrette_words_after_second_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): {len(unique_chastrette_only_after_second_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_second_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): 49
Remaining unique descriptors (first 10): ['coumarin', 'menthol', 'caramel', 'milk', 'pesticide', 'soap', 'beet', 'vinyl', 'cotton', 'bug']


Separating the odor descriptors in columns based in all the labels available in `smell_descriptors` list

In [ ]:
for descriptor in smell_descriptors:
    df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)

/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [ ]:
df.head()

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,C1=CC=C(C=C1)CCC(=O)O,balsamic,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(=O)O,sour,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC=O,pungent ethereal,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C(=O)C)O,creamy buttery,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC=C(C=C1)C=O,burnt almond,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("flavornet_processed.csv", index=False)

## Goodscents

In [ ]:
df = pd.from_csv("goodscents.csv")

## Leffingwell

In [ ]:
import ast
import pandas as pd

df = pd.read_csv("leffingwell_data.csv")
df = df[["smiles", "odor_labels_filtered"]]
df = df.rename(columns={'smiles': 'SMILES', 'odor_labels_filtered': 'Descriptors'})
df['Descriptors'] = df['Descriptors'].apply(ast.literal_eval)
df

,SMILES,Descriptors
0,CCOC(C)OCC,"[green, fruity, alcoholic]"
1,CCCCCOC(C)OCCC(C)C,"[vegetable, alcoholic, ethereal, green, cognac]"
2,COC(C)OC(C)OCc1ccccc1,"[green, fruity, almond, sweet]"
3,CCCCOC(C)OCC,"[alcoholic, ethereal, green, fruity, cognac]"
4,CCCCOC(C)OCCC(C)C,"[vegetable, alcoholic, ethereal, green, cognac]"
...,...,...
3518,CCCCC(C=O)SC(C)=O,"[grapefruit, sweet, citrus]"
3519,CCCCC(O)S,"[citrus, green, catty, brothy, grapefruit]"
3520,CC(O)C1CCCCC1,"[oily, fruity, fermented, herbal]"
3521,C=C1C(OC(C)=O)CC2CC1C2(C)C,"[fruity, camphoreous, mint, herbal]"


Correcting the little typos in the descriptors to match the correct ones that are present in the `smell_descriptors` list

In [ ]:
all_chastrette_words = df['Descriptors'].explode().unique().tolist()
print(f"Total unique descriptors from Leffingwell data: {len(all_chastrette_words)}")
print(f"First 10 unique descriptors: {all_chastrette_words[:10]}")

Total unique descriptors from Leffingwell data: 113
First 10 unique descriptors: ['green', 'fruity', 'alcoholic', 'vegetable', 'ethereal', 'cognac', 'almond', 'sweet', 'fermented', 'pungent']


In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 108
Common descriptors (first 10): ['medicinal', 'anisic', 'fresh', 'creamy', 'brandy', 'herbal', 'honey', 'lemon', 'vegetable', 'coconut']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Descriptors but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Descriptors (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors: 5
Descriptors unique to Descriptors (first 10): ['black currant', 'bread', 'jasmine', 'rum', 'mint']


Manually adding transformations for the remaining terms

In [ ]:
manual_transformations = {
    'black currant': 'currant black currant',
    'bread': 'bready',
    'jasmine': 'jasmin',
    'rum': 'rummy',
    'mint': 'minty'
}

transformations = {}

# Update the main transformations dictionary with these manual ones
transformations.update(manual_transformations)

# Redefine apply_transformations to work with lists of words directly
def apply_transformations(words_list, transformation_map):
    if not isinstance(words_list, list):
        # If it's not a list, it might be a single string from some previous context;
        # handle as a single string by wrapping it in a list for processing
        if isinstance(words_list, str):
            words_list = [words_list]
        else:
            return words_list # Return as is if neither a string nor a list

    transformed_words = []
    for word in words_list:
        # Ensure the word itself is a string before looking it up
        if isinstance(word, str):
            transformed_words.append(transformation_map.get(word, word))
        else:
            transformed_words.append(word) # Keep non-string elements as is
    return transformed_words

# Apply the transformations to the 'Descriptors' column
df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after manual transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after manual transformations:
0                         [green, fruity, alcoholic]
1    [vegetable, alcoholic, ethereal, green, cognac]
2                     [green, fruity, almond, sweet]
3       [alcoholic, ethereal, green, fruity, cognac]
4    [vegetable, alcoholic, ethereal, green, cognac]
Name: Descriptors, dtype: object


In [ ]:
for descriptor in smell_descriptors:
  for i in range(len(df)):
    if descriptor in df.at[i, 'Descriptors']:
      df.at[i, descriptor] = 1
    else:
      df.at[i, descriptor] = 0

In [ ]:
df.head()

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CCOC(C)OCC,"[green, fruity, alcoholic]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CCCCCOC(C)OCCC(C)C,"[vegetable, alcoholic, ethereal, green, cognac]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,COC(C)OC(C)OCc1ccccc1,"[green, fruity, almond, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CCCCOC(C)OCC,"[alcoholic, ethereal, green, fruity, cognac]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CCCCOC(C)OCCC(C)C,"[vegetable, alcoholic, ethereal, green, cognac]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("leffingwell_processed.csv", index=False)

## Sharma_2021b

In [ ]:
import pandas as pd

df_molecules = pd.read_csv("molecules_sharma2021a.csv")
df_molecules = df_molecules[["CID", "IsomericSMILES"]]
df_molecules.head()

,CID,IsomericSMILES
0,4,CC(CN)O
1,49,CC(C)C(=O)C(=O)O
2,58,CCC(=O)C(=O)O
3,70,CC(C)CC(=O)C(=O)O
4,72,C1=CC(=C(C=C1C(=O)O)O)O


In [ ]:
df_behavior = pd.read_csv("behavior_sharma2021a.csv")[['Stimulus', 'Decriptors']]
df_behavior = df_behavior.rename(columns={'Decriptors': 'Descriptors'})
df_behavior.head()

,Stimulus,Descriptors
0,4.0,fishy;ammonical
1,49.0,fruity
2,58.0,sweet;brown;lactonic;caramel;creamy
3,70.0,fruity
4,72.0,balsamic;phenolic


In [ ]:
df = pd.merge(df_behavior, df_molecules, left_on='Stimulus', right_on='CID', how='left')
df = df.rename(columns={'IsomericSMILES': 'SMILES'})
df = df.drop(columns=['CID', 'Stimulus'])
df = df[["SMILES", "Descriptors"]]
df = df.dropna().reset_index(drop=True)
df

,SMILES,Descriptors
0,CC(CN)O,fishy;ammonical
1,CC(C)C(=O)C(=O)O,fruity
2,CCC(=O)C(=O)O,sweet;brown;lactonic;caramel;creamy
3,CC(C)CC(=O)C(=O)O,fruity
4,C1=CC(=C(C=C1C(=O)O)O)O,balsamic;phenolic
...,...,...
3992,CCCC(=O)CC(=O)O.C(C(CO)O)O,fatty
3993,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,umami;meaty;salty
3994,CC(C)CC1CCCC(C1)(C)O.CC(C)CC1CCCC(C1)(C)O,floral;citrus;muguet
3995,CC1N=C(SC(S1)C)C.CC1N=C(SC(S1)C)C,beefy;meaty


Correcting the little typos in the descriptors to match the correct ones that are present in the `smell_descriptors` list

In [ ]:
all_chastrette_words = df['Descriptors'].str.split(';').explode().unique().tolist()
len(all_chastrette_words)

572

In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 399
Common descriptors (first 10): ['kimchi', 'ozone', 'vetiver', 'fig', 'lemon', 'vegetable', 'cooked', 'coconut', 'greasy', 'palmarosa']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Descriptors but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Descriptors (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors: 173
Descriptors unique to Descriptors (first 10): ['caramel', 'isojasmone', 'pepper', 'insipid', 'parmesan', 'citral', 'lie', 'calm', 'rotten fish', 'new mown hay']


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 90

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 51 potential transformations (threshold >= 90%):
  'pepper' -> 'peppery'
  'parmesan' -> 'cheesy parmesan cheese'
  'new mown hay' -> 'hay new mown hay'
  'green pear' -> 'green'
  'cheese' -> 'cheesy bleu cheese'
  'old wood' -> 'woody old wood'
  'cress' -> 'watercress'
  'limburger' -> 'cheesy limburger cheese'
  'bell pepper' -> 'pepper bell pepper'
  'deep fried' -> 'fried'


In [ ]:
df

,SMILES,Descriptors
0,CC(CN)O,fishy;ammonical
1,CC(C)C(=O)C(=O)O,fruity
2,CCC(=O)C(=O)O,sweet;brown;lactonic;caramel;creamy
3,CC(C)CC(=O)C(=O)O,fruity
4,C1=CC(=C(C=C1C(=O)O)O)O,balsamic;phenolic
...,...,...
3992,CCCC(=O)CC(=O)O.C(C(CO)O)O,fatty
3993,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,umami;meaty;salty
3994,CC(C)CC1CCCC(C1)(C)O.CC(C)CC1CCCC(C1)(C)O,floral;citrus;muguet
3995,CC1N=C(SC(S1)C)C.CC1N=C(SC(S1)C)C,beefy;meaty


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split(';')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after transformations:
0                       fishy ammoniacal
1                                 fruity
2    sweet brown lactonic caramel creamy
3                                 fruity
4                      balsamic phenolic
Name: Descriptors, dtype: object


Re-evaluating the unique words in `Descriptors` and comparing them again with `smell_descriptors`

In [ ]:
all_chastrette_words_after_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_transform = list(set(all_chastrette_words_after_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after transformations): {len(unique_chastrette_only_after_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after transformations): 172
Remaining unique descriptors (first 10): ['caramel', 'tar', 'isojasmone', 'pepper', 'insipid', 'parmesan', 'fruit', 'citral', 'lie', 'calm']


Re-applying `apply_transformations` function to the `Descriptors` column with the updated `transformations`

In [ ]:
df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

all_chastrette_words_after_second_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_second_transform = list(set(all_chastrette_words_after_second_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): {len(unique_chastrette_only_after_second_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_second_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): 172
Remaining unique descriptors (first 10): ['caramel', 'tar', 'isojasmone', 'pepper', 'insipid', 'parmesan', 'fruit', 'citral', 'lie', 'calm']


Separating the odor descriptors in columns based in all the labels available in `smell_descriptors` list

In [ ]:
for descriptor in smell_descriptors:
    df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)

/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [ ]:
df.head()

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,fishy ammoniacal,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(C)C(=O)C(=O)O,fruity,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CCC(=O)C(=O)O,sweet brown lactonic caramel creamy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)CC(=O)C(=O)O,fruity,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC(=C(C=C1C(=O)O)O)O,balsamic phenolic,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("sharma2021a_processed.csv", index=False)

## Sharma_2021b

In [ ]:
import pandas as pd

df_molecules = pd.read_csv("molecules_sharma2021b.csv")
df_molecules = df_molecules[["CID", "IsomericSMILES"]]
df_molecules.head()

,CID,IsomericSMILES
0,4,CC(CN)O
1,19,C1=CC(=C(C(=C1)O)O)C(=O)O
2,49,CC(C)C(=O)C(=O)O
3,51,C(CC(=O)O)C(=O)C(=O)O
4,58,CCC(=O)C(=O)O


In [ ]:
df_behavior = pd.read_csv("behavior_sharma2021b.csv")
df_behavior['Descriptors'] = df_behavior['Primary Odor'].fillna('') + '/' + df_behavior['Sub Odor'].fillna('')
df_behavior['Descriptors'] = df_behavior['Descriptors'].str.strip('/')
df_behavior = df_behavior.drop(columns=['Primary Odor', 'Sub Odor', 'Strength'])
df_behavior.head()

,Stimulus,Descriptors
0,4,Ammonia/Fishy
1,4,Fishy/Ammonia/Fishy
2,4,Fishy/Rancid/Fishy
3,4,Food/Fishy
4,19,Odorless


In [ ]:
df = pd.merge(df_behavior, df_molecules, left_on='Stimulus', right_on='CID', how='left')
df = df.rename(columns={'IsomericSMILES': 'SMILES'})
df = df.drop(columns=['CID', 'Stimulus'])
df = df[["SMILES", "Descriptors"]]
df = df.dropna().reset_index(drop=True)
df['Descriptors'] = df['Descriptors'].str.lower()
df

,SMILES,Descriptors
0,CC(CN)O,ammonia/fishy
1,CC(CN)O,fishy/ammonia/fishy
2,CC(CN)O,fishy/rancid/fishy
3,CC(CN)O,food/fishy
4,C1=CC(=C(C(=C1)O)O)C(=O)O,odorless
...,...,...
54227,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless
54228,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless
54229,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless
54230,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless


Correcting the little typos in the descriptors to match the correct ones that are present in the `smell_descriptors` list

In [ ]:
all_chastrette_words = df['Descriptors'].str.split('/').explode().unique().tolist()
len(all_chastrette_words)

544

In [ ]:
common_descriptors = list(set(all_chastrette_words).intersection(set(smell_descriptors)))
print(f"Number of common descriptors: {len(common_descriptors)}")
print(f"Common descriptors (first 10): {common_descriptors[:10]}")

Number of common descriptors: 321
Common descriptors (first 10): ['kimchi', 'ozone', 'vetiver', 'herbal', 'lemon', 'fig', 'vegetable', 'cooked', 'palmarosa', 'hyacinth']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Descriptors but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Descriptors (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors: 223
Descriptors unique to Descriptors (first 10): ['waste', 'caramel', 'people&animals', 'pepper', 'sweet tobacco', 'microbial', 'citral', 'clothes', 'ozonous', "'grassy (fresh, sweet)'"]


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 75

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 123 potential transformations (threshold >= 75%):
  'caramel' -> 'caramellic'
  'people&animals' -> 'animal'
  'pepper' -> 'peppery'
  'sweet tobacco' -> 'sweet'
  ''grassy (fresh, sweet)'' -> 'fresh'
  'pulpy fruit' -> 'pulpy'
  'rotten fish' -> 'fishy'
  'new mown hay' -> 'hay new mown hay'
  'limburger cheese' -> 'cheesy limburger cheese'
  'alcohol' -> 'alcoholic'


In [ ]:
df

,SMILES,Descriptors
0,CC(CN)O,ammonia/fishy
1,CC(CN)O,fishy/ammonia/fishy
2,CC(CN)O,fishy/rancid/fishy
3,CC(CN)O,food/fishy
4,C1=CC(=C(C(=C1)O)O)C(=O)O,odorless
...,...,...
54227,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless
54228,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless
54229,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless
54230,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split('/')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after transformations:
0          ammoniacal fishy
1    fishy ammoniacal fishy
2        fishy rancid fishy
3             seafood fishy
4                  odorless
Name: Descriptors, dtype: object


Re-evaluating the unique words in `Descriptors` and comparing them again with `smell_descriptors`

In [ ]:
all_chastrette_words_after_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_transform = list(set(all_chastrette_words_after_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after transformations): {len(unique_chastrette_only_after_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after transformations): 144
Remaining unique descriptors (first 10): ['waste', 'parmesan', 'pepper', 'microbial', 'fruit', 'citral', 'clothes', 'ozonous', 'sauce', 'needle']


Re-applying `apply_transformations` function to the `Descriptors` column with the updated `transformations`

In [ ]:
df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

all_chastrette_words_after_second_transform = df['Descriptors'].str.split(' ').explode().unique().tolist()

unique_chastrette_only_after_second_transform = list(set(all_chastrette_words_after_second_transform) - set(smell_descriptors))

print(f"Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): {len(unique_chastrette_only_after_second_transform)}")
print(f"Remaining unique descriptors (first 10): {unique_chastrette_only_after_second_transform[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors (after second round of transformations): 144
Remaining unique descriptors (first 10): ['waste', 'parmesan', 'pepper', 'microbial', 'fruit', 'citral', 'clothes', 'ozonous', 'sauce', 'needle']


Separating the odor descriptors in columns based in all the labels available in `smell_descriptors` list

In [ ]:
for descriptor in smell_descriptors:
    df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)

/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[descriptor] = df['Descriptors'].str.contains(descriptor, case=False, na=False).astype(int)
/tmp/ipykernel_2091/3242236013.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [ ]:
df

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,ammoniacal fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(CN)O,fishy ammoniacal fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC(CN)O,fishy rancid fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(CN)O,seafood fishy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC(=C(C(=C1)O)O)C(=O)O,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54227,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
54228,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
54229,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
54230,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,odorless,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("sharma2021b_processed.csv", index=False)

## Sigma_2014

In [ ]:
df_molecules = pd.read_csv("molecules_sigma2014.csv")
df_molecules = df_molecules[["CID", "IsomericSMILES"]]
df_molecules

,CID,IsomericSMILES
0,58,CCC(=O)C(=O)O
1,107,C1=CC=C(C=C1)CCC(=O)O
2,126,C1=CC(=CC=C1C=O)O
3,135,C1=CC(=CC=C1C(=O)O)O
4,176,CC(=O)O
...,...,...
862,44134940,C1=CC=C2C(=C1)C(=O)[N-]S2(=O)=O.C1=CC=C2C(=C1)...
863,54675810,C([C@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O
864,55251157,CC1=CN=C(C(=N1)C)SC
865,91001317,CC.OO


In [ ]:
df_behavior = pd.read_csv("behavior_sigma2014.csv")
df_behavior = df_behavior[['Stimulus', 'descriptors']]
df_behavior = df_behavior.rename(columns={'descriptors': 'Descriptors'})
df_behavior = df_behavior.iloc[69:-1].reset_index(drop=True)
df_behavior.head()

,Stimulus,Descriptors
0,58,"['caramel', 'creamy']"
1,107,"['rose', 'sweet']"
2,126,"['lily', 'sweet']"
3,135,['caramel']
4,176,"['apple', 'ethereal']"


In [ ]:
df = pd.merge(df_behavior, df_molecules, left_on='Stimulus', right_on='CID', how='left')
df = df.rename(columns={'IsomericSMILES': 'SMILES'})
df = df.drop(columns=['CID', 'Stimulus'])
df = df[["SMILES", "Descriptors"]]
df['Descriptors'] = df['Descriptors'].apply(ast.literal_eval)
df = df.dropna().reset_index(drop=True)
df

,SMILES,Descriptors
0,CCC(=O)C(=O)O,"[caramel, creamy]"
1,C1=CC=C(C=C1)CCC(=O)O,"[rose, sweet]"
2,C1=CC(=CC=C1C=O)O,"[lily, sweet]"
3,C1=CC(=CC=C1C(=O)O)O,[caramel]
4,CC(=O)O,"[apple, ethereal]"
...,...,...
866,CC1C(C(C(C(O1)CO)O)OC2C(C(C(C(O2)COC3C(C(C(CO3...,[spicy]
867,C1=CC=C2C(=C1)C(=O)[N-]S2(=O)=O.C1=CC=C2C(=C1)...,"[lemon, musty, oily, fruity, rose, sweet]"
868,C([C@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O,"[vanilla, woody]"
869,CC1=CN=C(C(=N1)C)SC,"[almond, caramel, musty, vegetable]"


In [ ]:
all_chastrette_words = df['Descriptors'].explode().unique().tolist()
print(f"Total unique descriptors from Leffingwell data: {len(all_chastrette_words)}")
print(f"First 10 unique descriptors: {all_chastrette_words[:10]}")

Total unique descriptors from Leffingwell data: 116
First 10 unique descriptors: ['caramel', 'creamy', 'rose', 'sweet', 'lily', 'apple', 'ethereal', 'almond', 'cherry', 'balsam']


In [ ]:
unique_chastrette_only = list(set(all_chastrette_words) - set(smell_descriptors))
print(f"Number of descriptors in Descriptors but not in smell_descriptors: {len(unique_chastrette_only)}")
print(f"Descriptors unique to Descriptors (first 10): {unique_chastrette_only[:10]}")

Number of descriptors in Descriptors but not in smell_descriptors: 21
Descriptors unique to Descriptors (first 10): ['coumarin', 'caramel', 'camphoraceous', 'pepper', 'hawthorne', 'herbaceous', 'winelike', 'alliaceous (onion, garlic)', 'alcohol', 'jasmine']


In [ ]:
from fuzzywuzzy import process

SIMILARITY_THRESHOLD = 70

transformations = {}
for unique_word in unique_chastrette_only:
    if not unique_word.strip():
        continue

    best_match, score = process.extractOne(unique_word, smell_descriptors)

    if score >= SIMILARITY_THRESHOLD:
        transformations[unique_word] = best_match

print(f"Found {len(transformations)} potential transformations (threshold >= {SIMILARITY_THRESHOLD}%):")
for original, transformed in list(transformations.items())[:10]:
    print(f"  '{original}' -> '{transformed}'")

Found 19 potential transformations (threshold >= 70%):
  'coumarin' -> 'coumarinic'
  'caramel' -> 'caramellic'
  'camphoraceous' -> 'camphoreous'
  'pepper' -> 'peppery'
  'hawthorne' -> 'hawthorn'
  'herbaceous' -> 'herbal'
  'winelike' -> 'winey'
  'alliaceous (onion, garlic)' -> 'alliaceous'
  'alcohol' -> 'alcoholic'
  'jasmine' -> 'jasmin'


In [ ]:
def apply_transformations(text, transformation_map):
    if not isinstance(text, str):
        return text

    words = text.split(' ')
    transformed_words = []
    for word in words:
        transformed_words.append(transformation_map.get(word, word))
    return ' '.join(transformed_words)

df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after transformations:
0    [caramel, creamy]
1        [rose, sweet]
2        [lily, sweet]
3            [caramel]
4    [apple, ethereal]
Name: Descriptors, dtype: object


In [ ]:
# manual_transformations = {
#     'black currant': 'currant black currant',
#     'bread': 'bready',
#     'jasmine': 'jasmin',
#     'rum': 'rummy',
#     'mint': 'minty'
# }

# Update the main transformations dictionary with these manual ones
transformations.update(manual_transformations)

# Redefine apply_transformations to work with lists of words directly
def apply_transformations(words_list, transformation_map):
    if not isinstance(words_list, list):
        # If it's not a list, it might be a single string from some previous context;
        # handle as a single string by wrapping it in a list for processing
        if isinstance(words_list, str):
            words_list = [words_list]
        else:
            return words_list # Return as is if neither a string nor a list

    transformed_words = []
    for word in words_list:
        # Ensure the word itself is a string before looking it up
        if isinstance(word, str):
            transformed_words.append(transformation_map.get(word, word))
        else:
            transformed_words.append(word) # Keep non-string elements as is
    return transformed_words

# Apply the transformations to the 'Descriptors' column
df['Descriptors'] = df['Descriptors'].apply(lambda x: apply_transformations(x, transformations))

print("First 5 rows of 'Descriptors' after manual transformations:")
print(df['Descriptors'].head())

First 5 rows of 'Descriptors' after manual transformations:
0    [caramellic, creamy]
1           [rose, sweet]
2           [lily, sweet]
3            [caramellic]
4       [apple, ethereal]
Name: Descriptors, dtype: object


In [ ]:
for descriptor in smell_descriptors:
  for i in range(len(df)):
    if descriptor in df.at[i, 'Descriptors']:
      df.at[i, descriptor] = 1
    else:
      df.at[i, descriptor] = 0

/tmp/ipykernel_2091/2741500261.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[i, descriptor] = 0
/tmp/ipykernel_2091/2741500261.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.at[i, descriptor] = 0
/tmp/ipykernel_2091/2741500261.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df

In [ ]:
for col in smell_descriptors:
    if col in df.columns:
        df[col] = df[col].astype(int)
df

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CCC(=O)C(=O)O,"[caramellic, creamy]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C1=CC=C(C=C1)CCC(=O)O,"[rose, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C1=CC(=CC=C1C=O)O,"[lily, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C1=CC(=CC=C1C(=O)O)O,[caramellic],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CC(=O)O,"[apple, ethereal]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,CC1C(C(C(C(O1)CO)O)OC2C(C(C(C(O2)COC3C(C(C(CO3...,[spicy],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
867,C1=CC=C2C(=C1)C(=O)[N-]S2(=O)=O.C1=CC=C2C(=C1)...,"[lemon, musty, oily, fruity, rose, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
868,C([C@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O,"[vanilla, woody]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
869,CC1=CN=C(C(=N1)C)SC,"[almond, caramellic, musty, vegetable]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.head()

,SMILES,Descriptors,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CCC(=O)C(=O)O,"[caramellic, creamy]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C1=CC=C(C=C1)CCC(=O)O,"[rose, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C1=CC(=CC=C1C=O)O,"[lily, sweet]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C1=CC(=CC=C1C(=O)O)O,[caramellic],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CC(=O)O,"[apple, ethereal]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Saving the processed dataset

In [ ]:
df.to_csv("sigma2014_processed.csv", index=False)

# Combining all the datasets

To do this, I'm gonna start from the Good Scents dataset, and then merge everything one by one

In [3]:
import pandas as pd

df = pd.read_csv("goodscents_processed.csv")
df = df.drop(columns=['Descriptors'])
df.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,C1C=CC2C1C3CC2CC3OCC=O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC[C@H]1CC(CCC1=O)(C)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC(=C)[C@@H]1CC[C@]2([C@@H](C1)O2)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC[C@H](C)C(=O)/C=C/C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CC[C@]1([C@@]2(CC[C@@H](C2)C1(C)C)C)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `arctander` dataset

In [4]:
df_aux = pd.read_csv("arctander_processed.csv")
df_aux = df_aux.drop(columns=['ChastretteDetails'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(C)C1CCC2C(=C1)CCC3C2(CCCC3(C)CO)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C(C)=O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC(OCCOC)OCC1=CC=CC=C1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)CCOC(C)OCCC(C)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CCOC(C)OCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (7175, 584)


In [6]:
df.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B1(OB2OB(OB(O1)O2)[O-])[O-].O.O.O.O.O.O.O.O.O....,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,BrC1=C(C=CC=C1)CC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,BrC=CC1=CC=CC=C1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCCCCCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `leffingwell` dataset

In [7]:
df_aux = pd.read_csv("leffingwell_processed.csv")
df_aux = df_aux.drop(columns=['Descriptors'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CCOC(C)OCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CCCCCOC(C)OCCC(C)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,COC(C)OC(C)OCc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CCCCOC(C)OCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CCCCOC(C)OCCC(C)C,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (9957, 584)


/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

In [9]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B1(OB2OB(OB(O1)O2)[O-])[O-].O.O.O.O.O.O.O.O.O....,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,BrC1=C(C=CC=C1)CC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,BrC=CC1=CC=CC=C1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCCCCCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9952,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9953,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9954,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9955,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `sharma_2021a` dataset

In [10]:
df_aux = pd.read_csv("sharma2021a_processed.csv")
df_aux = df_aux.drop(columns=['Descriptors'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(C)C(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CCC(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)CC(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC(=C(C=C1C(=O)O)O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (10474, 584)


In [12]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B1(OB2OB(OB(O1)O2)[O-])[O-].O.O.O.O.O.O.O.O.O....,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,BrC1=C(C=CC=C1)CC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,BrC=CC1=CC=CC=C1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCCCCCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10469,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10470,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10471,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10472,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `sigma_2014` dataset

In [13]:
df_aux = pd.read_csv("sigma2014_processed.csv")
df_aux = df_aux.drop(columns=['Descriptors'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CCC(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C1=CC=C(C=C1)CCC(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C1=CC(=CC=C1C=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C1=CC(=CC=C1C(=O)O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CC(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (10507, 584)


In [15]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B1(OB2OB(OB(O1)O2)[O-])[O-].O.O.O.O.O.O.O.O.O....,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,BrC1=C(C=CC=C1)CC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,BrC=CC1=CC=CC=C1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCCCCCC,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10502,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10503,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10504,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10505,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **From now on, I don't know if its worth the merge**

## Merging with the `flavordb` dataset

Since it seems this dataset accounts for flavors, I don't know if its worth merging with it. In the pyrfurme git repo, its marked as odor related

In [16]:
df_aux = pd.read_csv("flavordb_processed.csv")
df_aux = df_aux.drop(columns=['Flavor Percepts'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C1C(C(C(OC1O)CO)O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CCC(C)C(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C)C(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C(CC(=O)O)C(=O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (33319, 584)


In [18]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B(C1=CC2=C(C=C1)S(=O)(=O)NC2=O)(O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,B(CCOC[C@H]1[C@H]([C@H]([C@@H](C(O1)O)O)O)O)(O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,B(O)(O)OC(C(CO)O)C(C(CO)O)OC1C(C(C(C(O1)CO)O)O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,B(O)(O)OC1C(C(OC1(CO)OC2C(C(C(C(O2)CO)O)O)O)CO)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33314,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33315,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33316,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33317,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `flavornet` dataset

Since it seems this dataset accounts for flavors aswell, I don't know if its worth merging with it. In the pyrfurme git repo, its marked as odor related

In [19]:
df_aux = pd.read_csv("flavornet_processed.csv")
df_aux = df_aux.drop(columns=['Descriptors'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,C1=CC=C(C=C1)CCC(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC=O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(C(=O)C)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC=C(C=C1)C=O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (33338, 584)


In [21]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B(C1=CC2=C(C=C1)S(=O)(=O)NC2=O)(O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,B(CCOC[C@H]1[C@H]([C@H]([C@@H](C(O1)O)O)O)O)(O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,B(O)(O)OC(C(CO)O)C(C(CO)O)OC1C(C(C(C(O1)CO)O)O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,B(O)(O)OC1C(C(OC1(CO)OC2C(C(C(C(O2)CO)O)O)O)CO)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33333,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33334,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33335,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
33336,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Merging with the `sharma_2021b` dataset

This dataset is huge (more than 50k molecules), and I don't know if its origin is trustable (I prefer to leave this one as the last one to merge)

In [22]:
df_aux = pd.read_csv("sharma2021b_processed.csv")
df_aux = df_aux.drop(columns=['Descriptors'])

df_aux.head()

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CC(CN)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C1=CC(=C(C(=C1)O)O)C(=O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
descriptor_cols = [col for col in df.columns if col != 'SMILES']

merged_df = pd.merge(df, df_aux, on='SMILES', how='outer', suffixes=('_df', '_df_aux'))

for col in descriptor_cols:
    merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)

cols_to_drop = [col for col in merged_df.columns if col.endswith('_df') or col.endswith('_df_aux')]
merged_df = merged_df.drop(columns=cols_to_drop)

print(f"\nShape of the merged DataFrame: {merged_df.shape}")

df = merged_df.copy()

/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df[col] = ((merged_df[f'{col}_df'].fillna(0) + merged_df[f'{col}_df_aux'].fillna(0)) > 0).astype(int)
/tmp/ipykernel_9444/2385381530.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti


Shape of the merged DataFrame: (86715, 584)


In [24]:
df

,SMILES,absinthe,acacia,acai,acerola,acetic,acetone,acidic,acorn,acrylate,...,woody burnt wood,woody oak wood,woody old wood,wormwood,yeasty,ylang,yogurt,yuzu,zedoary,zesty
0,B(=O)OB=O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,B(=O)O[O-].O.[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,B(=O)O[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,B(=O)[O-].[Na+],0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,B(C1=CC2=C(C=C1)S(=O)(=O)NC2=O)(O)O,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86710,c1coc(CSSCc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86711,c1coc(Cc2ccoc2Cc2ccco2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86712,c1coc(Cn2cccc2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86713,c1csc(SSc2cccs2)c1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Saving the final dataset**

In [25]:
df.to_csv("final_odor_dataset.csv", index=False)

# Checking if one specific molecules has already been added to the dataset

In [26]:
import pandas as pd

df_final = pd.read_csv("final_odor_dataset.csv")

In [27]:
def check_molecule_in_dataset(molecule, df):
  is_in = False

  for i in range(len(df["SMILES"])):
    if df['SMILES'][i] == molecule:
      print("The molecule", df['SMILES'][i], "is part of the dataset.")
      is_in = True

      break

  if(not is_in):
    print("The molecule", molecule, "is not part of the dataset.")

## Testing the 6 molecules we have in our dataset

In [28]:
df_test = pd.read_csv("e_nose_data_complete.csv")[["SMILES", "cleaned_item"]].iloc[:-3]
unique_smiles = df_test['SMILES'].unique()
unique_names = df_test['cleaned_item'].unique()
print(unique_smiles)
print(unique_names)

['CC(C)/C=C/C=C(\\C)/C=C' 'CC1=CCC2C(C1)C2(C)C' 'CC(=CCCC(C)(C=C)O)C'
 'CC1=C[C@H]2C[C@@H](C1)C2(C)C' 'CC1=CC[C@H](CC1)C(=C)C'
 'CC1=CC[C@@H](CC1)C(=C)C']
['ocimene' 'delta-3-carene' 'linalol' 'a-pinene' 's-limonene' 'r-limonene']


Ocimene

In [29]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[0])], df_final)

The molecule CC(C)/C=C/C=C(\C)/C=C is not part of the dataset.


Delta-3-carene

In [30]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[1])], df_final)

The molecule CC1=CCC2C(C1)C2(C)C is part of the dataset.


Linalol

In [31]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[2])], df_final)

The molecule CC(=CCCC(C)(C=C)O)C is part of the dataset.


A-pinene

In [32]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[3])], df_final)

The molecule CC1=C[C@H]2C[C@@H](C1)C2(C)C is not part of the dataset.


S-limonene

In [33]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[4])], df_final)

The molecule CC1=CC[C@H](CC1)C(=C)C is part of the dataset.


R-limonene

In [34]:
check_molecule_in_dataset(unique_smiles[list(unique_names).index(unique_names[5])], df_final)

The molecule CC1=CC[C@@H](CC1)C(=C)C is part of the dataset.
